In [1]:
from transformers import AutoTokenizer
import torch

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [3]:
word = "Beautiful"
out = tokenizer(word,return_tensors= "pt").input_ids

In [4]:
out

tensor([[38413,  4135]])

In [5]:
word = "pneumonoultramicroscopicsilicovolcanoconiosis"
out = tokenizer(word,return_tensors = "pt").input_ids

for token in out[0]:
    print(tokenizer.decode(token))

p
neum
on
oult
ram
icro
sc
op
ics
ilic
ov
ol
can
ocon
iosis


In [6]:
word = "Hippopotomonstrosesquippedaliophobia"
out = tokenizer(word,return_tensors="pt").input_ids

for token in out[0]:
    print(tokenizer.decode(token))

H
ipp
opot
omon
stros
es
qu
ipped
ali
ophobia


## AutoModel usecase

In [7]:
from transformers import AutoModelForCausalLM

gpt_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt_model

D:\Laptop Backup\Softpro\Data Analytics\myenv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [49]:
sentence = "I love Machine Learning to enhance my work for the benefit of others"
tokens = tokenizer(sentence,return_tensors= "pt").input_ids
next_token = gpt_model(tokens).logits[0,-1]
tokenizer.decode(next_token.argmax())

'.'

In [37]:
sentence = "I Learn Machine Learning to enhance productivity"
tokens = tokenizer(sentence,return_tensors = "pt").input_ids
next_token = gpt_model(tokens).logits.squeeze()[-1]
# returns the top k values to be predicted as next token by gpt model
out = torch.topk(next_token,k=10)

In [50]:
out.indices

tensor([   13,   198,   290,    11,   416,   287,   553,   351, 50256,    25])

In [38]:
for indices in out.indices:
    print(tokenizer.decode(indices))

.


 and
,
 by
 in
,"
 with
<|endoftext|>
:


In [12]:
torch.softmax(out.values,dim = 0)

tensor([0.3579, 0.2380, 0.1013, 0.0973, 0.0594, 0.0514, 0.0321, 0.0267, 0.0183,
        0.0176], grad_fn=<SoftmaxBackward0>)

In [51]:
def top_k_sampling(logits,k = 50):

    values, indices = torch.topk(logits,k)
    probs = torch.softmax(values,dim = -1)
    samples = torch.multinomial(probs,5)
    return indices[samples]
    

In [23]:
torch.multinomial(torch.tensor([0.4,0.5,0.8,0.9,0.1,0.78,0.99]),3)

tensor([3, 2, 6])

In [52]:
tokenizer.decode(top_k_sampling(next_token))

'. and, as on'